In [1]:
from text_embedder import TextProteinQueryEncoder
from protein_dataset import TextProteinQueryDataset
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import h5py
import numpy as np


In [2]:
# Loads the model
model_name = "nomic-ai/nomic-embed-text-v1"
encoder = ProteinQueryEncoder(model_name)

Loading weights:   0%|          | 0/112 [00:00<?, ?it/s]

In [3]:
# Loads the dataset

dataset_name = "cell_texts.csv

dataset = ProteinQueryDataset(dataset_name, query_transform=lambda x: "search_query: "+x, in_memory=True)
loader = DataLoader(dataset, batch_size=64)

In [4]:
# Produces the embedding

cell_ids = []
embeddings = []

with torch.no_grad():
    for batch in tqdm(loader):
        queries, ids = batch
        emb = encoder(batch).cpu()
        
        cell_ids.extend(ids)
        embeddings.extend(emb)

100%|██████████| 2825/2825 [01:32<00:00, 30.50it/s]


In [21]:
# Dumps into the h5 file

embeddings_tensor = torch.stack(embeddings).numpy()
id_array = np.array(cell_ids, dtype=h5py.string_dtype()) 

with h5py.File("cell_embeddings.h5", "w") as f:
    f.create_dataset("embeddings", data=embeddings_tensor, compression="gzip")
    f.create_dataset("cell_ids", data=id_array)